In [1]:
import pandas as pd
import numpy as np
import time

file_path = '/content/clq26 (60 min tick by tick data).txt'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

df['DateTime'] = pd.to_datetime(
    df['Date'].astype(str).str.strip() + ' ' +
    df['Time'].astype(str).str.strip(),
    errors='coerce'
)

df = df.rename(columns={'Last': 'price'})
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['DateTime', 'price'])
df = df[df['price'] > 0].sort_values('DateTime').reset_index(drop=True)

print(f"Raw ticks loaded: {len(df):,}")
print(f"Date range: {df['DateTime'].iloc[0]} → {df['DateTime'].iloc[-1]}")

# Resample to 60-min OHLC bars
df = df.set_index('DateTime')
bars = df['price'].resample('60min').ohlc()
bars = bars.dropna().reset_index()

print(f"\nResampled to 60-min bars: {len(bars):,}")
print(f"Bar date range: {bars['DateTime'].iloc[0]} → {bars['DateTime'].iloc[-1]}")
bars.head()

Raw ticks loaded: 808,935
Date range: 2023-03-08 19:40:44.777000 → 2026-06-30 06:18:14.364000

Resampled to 60-min bars: 4,211
Bar date range: 2023-03-08 19:00:00 → 2026-06-30 06:00:00


,DateTime,open,high,low,close
0,2023-03-08 19:00:00,64.25,64.25,64.22,64.22
1,2024-07-30 15:00:00,67.68,67.68,67.68,67.68
2,2024-08-20 17:00:00,66.69,66.71,66.69,66.71
3,2024-09-10 17:00:00,63.28,63.28,63.28,63.28
4,2024-09-10 18:00:00,63.36,63.36,63.35,63.35


In [2]:
POINT_VALUE       = 1000
MAX_DD_USER_LIMIT = -10000   # change this to your actual limit

# SuperTrend search space
ST_ATR_PERIOD_RANGE = range(5, 31, 1)              # 5 to 30
ST_MULT_RANGE        = np.arange(1.5, 5.1, 0.25)   # 1.5 to 5.0

# Stop/Target search space (separate ATR for risk management)
STOPTARGET_ATR_PERIOD_RANGE = [7, 10, 14, 21]
STOP_MULT_RANGE   = np.arange(0.5, 3.1, 0.25)
TARGET_MULT_RANGE = np.arange(1.0, 6.1, 0.5)

close = bars['close'].values
high  = bars['high'].values
low   = bars['low'].values
n     = len(bars)
print(f"Total bars for optimization: {n:,}")

Total bars for optimization: 4,211


In [12]:
def calc_supertrend(high, low, close, atr_period, multiplier):
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(
            high[i] - low[i],
            abs(high[i] - close[i-1]),
            abs(low[i] - close[i-1])
        )
    atr = pd.Series(tr).rolling(atr_period).mean().values

    hl2 = (high + low) / 2.0
    raw_upper = hl2 + multiplier * atr
    raw_lower = hl2 - multiplier * atr

    final_upper = np.zeros(n)
    final_lower = np.zeros(n)
    direction   = np.ones(n, dtype=int)
    st_line     = np.zeros(n)

    first_valid = atr_period  # first index where atr is not NaN

    for i in range(n):
        if i < first_valid:
            # Not enough data yet — placeholder values, no signal
            final_upper[i] = hl2[i]
            final_lower[i] = hl2[i]
            direction[i] = 1
            st_line[i] = close[i]
            continue

        if i == first_valid:
            # First valid bar — initialize bands directly from raw values
            final_upper[i] = raw_upper[i]
            final_lower[i] = raw_lower[i]
            direction[i] = 1 if close[i] > raw_lower[i] else -1
            st_line[i] = final_lower[i] if direction[i] == 1 else final_upper[i]
            continue

        # Normal ratchet logic — prev values are guaranteed valid now
        if raw_upper[i] < final_upper[i-1] or close[i-1] > final_upper[i-1]:
            final_upper[i] = raw_upper[i]
        else:
            final_upper[i] = final_upper[i-1]

        if raw_lower[i] > final_lower[i-1] or close[i-1] < final_lower[i-1]:
            final_lower[i] = raw_lower[i]
        else:
            final_lower[i] = final_lower[i-1]

        if direction[i-1] == 1:
            direction[i] = -1 if close[i] < final_lower[i] else 1
        else:
            direction[i] = 1 if close[i] > final_upper[i] else -1

        st_line[i] = final_lower[i] if direction[i] == 1 else final_upper[i]

    return st_line, direction, atr

print("SuperTrend function ready (fixed NaN propagation bug)")

SuperTrend function ready (fixed NaN propagation bug)


In [13]:
def backtest_supertrend_atr(high, low, close,
                             st_period, st_mult,
                             stop_atr_period, stop_mult, target_mult,
                             dd_limit=None):

    st_line, direction, _ = calc_supertrend(high, low, close, st_period, st_mult)

    # Separate ATR for stop/target sizing
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i]-low[i], abs(high[i]-close[i-1]), abs(low[i]-close[i-1]))
    risk_atr = pd.Series(tr).rolling(stop_atr_period).mean().values

    trades = []
    pos    = 0
    ep     = 0.0
    stop   = 0.0
    target = 0.0
    cum    = 0.0
    pk     = 0.0
    max_dd = 0.0

    warmup = max(st_period, stop_atr_period) + 1

    for i in range(warmup, n):
        if np.isnan(st_line[i]) or np.isnan(risk_atr[i]):
            continue

        # Check exits first (race: ST flip, target, stop)
        if pos == 1:
            st_flip = (direction[i] == -1)
            hit_target = (high[i] >= target)
            hit_stop   = (low[i] <= stop)

            if st_flip or hit_target or hit_stop:
                if hit_stop and not hit_target:
                    exit_price = stop
                elif hit_target and not hit_stop:
                    exit_price = target
                elif hit_stop and hit_target:
                    exit_price = stop  # conservative: assume stop hit first
                else:
                    exit_price = close[i]  # ST flip exit at close

                pnl = (exit_price - ep) * POINT_VALUE
                trades.append(pnl)
                cum += pnl
                pos = 0

        elif pos == -1:
            st_flip = (direction[i] == 1)
            hit_target = (low[i] <= target)
            hit_stop   = (high[i] >= stop)

            if st_flip or hit_target or hit_stop:
                if hit_stop and not hit_target:
                    exit_price = stop
                elif hit_target and not hit_stop:
                    exit_price = target
                elif hit_stop and hit_target:
                    exit_price = stop
                else:
                    exit_price = close[i]

                pnl = (ep - exit_price) * POINT_VALUE
                trades.append(pnl)
                cum += pnl
                pos = 0

        # Entry on ST flip when flat
        if pos == 0:
            flipped_bull = (direction[i] == 1 and direction[i-1] == -1)
            flipped_bear = (direction[i] == -1 and direction[i-1] == 1)

            if flipped_bull:
                pos    = 1
                ep     = close[i]
                stop   = ep - stop_mult * risk_atr[i]
                target = ep + target_mult * risk_atr[i]
            elif flipped_bear:
                pos    = -1
                ep     = close[i]
                stop   = ep + stop_mult * risk_atr[i]
                target = ep - target_mult * risk_atr[i]

        # Track drawdown
        if pos == 1:
            opnl = (close[i] - ep) * POINT_VALUE
        elif pos == -1:
            opnl = (ep - close[i]) * POINT_VALUE
        else:
            opnl = 0.0

        run = cum + opnl
        if run > pk:
            pk = run
        dd = run - pk
        if dd < max_dd:
            max_dd = dd
        if dd_limit is not None and max_dd < dd_limit:
            return None

    if len(trades) < 3:
        return None

    pnls = np.array(trades)
    w = pnls[pnls > 0]
    l = pnls[pnls < 0]
    tp = w.sum() if len(w) else 0.0
    tl = abs(l.sum()) if len(l) else 1.0

    return {
        'st_period':   st_period,
        'st_mult':     round(st_mult, 2),
        'stop_atr_p':  stop_atr_period,
        'stop_mult':   round(stop_mult, 2),
        'target_mult': round(target_mult, 2),
        'pf':          round(tp / tl, 4) if tl > 0 else 999.0,
        'total_pnl':   round(float(pnls.sum()), 0),
        'trades':      len(pnls),
        'win_rate':    round(len(w) / len(pnls) * 100, 1),
        'avg_win':     round(float(w.mean()), 0) if len(w) else 0,
        'avg_loss':    round(float(l.mean()), 0) if len(l) else 0,
        'max_dd':      round(float(max_dd), 0),
    }

print("Backtest engine ready")

Backtest engine ready


In [14]:
t0 = time.time()

# Fixed reasonable stop/target to isolate SuperTrend quality first
FIXED_STOP_ATR_P = 14
FIXED_STOP_MULT  = 1.5
FIXED_TGT_MULT   = 3.0

st_combos = [(p, m) for p in ST_ATR_PERIOD_RANGE for m in ST_MULT_RANGE]
print(f"Stage 1: Testing {len(st_combos)} SuperTrend parameter combinations...")

st_results = []
for i, (p, m) in enumerate(st_combos):
    r = backtest_supertrend_atr(high, low, close, p, m,
                                 FIXED_STOP_ATR_P, FIXED_STOP_MULT, FIXED_TGT_MULT)
    if r:
        st_results.append(r)
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(st_combos)} tested | elapsed {time.time()-t0:.0f}s")

st_results.sort(key=lambda x: x['pf'], reverse=True)
print(f"\nTop 10 SuperTrend parameter sets (fixed stop/target):")
print(f"{'STPeriod':>9} {'STMult':>7} {'PF':>8} {'PnL':>12} {'Trades':>8} {'MaxDD':>12}")
for r in st_results[:10]:
    print(f"{r['st_period']:>9} {r['st_mult']:>7} {r['pf']:>8.4f} "
          f"${r['total_pnl']:>10,.0f} {r['trades']:>8} ${r['max_dd']:>10,.0f}")

best_st_period = st_results[0]['st_period']
best_st_mult   = st_results[0]['st_mult']
print(f"\nBest SuperTrend: Period={best_st_period}, Mult={best_st_mult}")
print(f"Elapsed: {time.time()-t0:.1f}s")

Stage 1: Testing 390 SuperTrend parameter combinations...
  50/390 tested | elapsed 2s
  100/390 tested | elapsed 3s
  150/390 tested | elapsed 5s
  200/390 tested | elapsed 6s
  250/390 tested | elapsed 8s
  300/390 tested | elapsed 10s
  350/390 tested | elapsed 13s

Top 10 SuperTrend parameter sets (fixed stop/target):
 STPeriod  STMult       PF          PnL   Trades        MaxDD
       13     3.5   1.3988 $    25,161      145 $   -10,632
       21     5.0   1.3956 $    13,831       79 $    -8,939
       23     4.0   1.3279 $    15,650      105 $    -8,836
       12     3.5   1.3065 $    19,465      149 $   -10,632
       20    4.75   1.2939 $    10,887       83 $   -11,618
       13     2.5   1.2895 $    25,583      239 $   -18,175
       23    3.75   1.2851 $    14,444      117 $    -8,811
       21    4.25   1.2840 $    13,043      101 $    -8,245
       11    4.25   1.2802 $    14,419      113 $   -11,431
       18     4.0   1.2789 $    14,281      115 $    -8,937

Best SuperTre

In [6]:
# Diagnostic — test a single combo and inspect why it fails
test_p, test_m = 10, 3.0
r = backtest_supertrend_atr(high, low, close, test_p, test_m, 14, 1.5, 3.0)
print("Result:", r)

# Manually check SuperTrend output
st_line, direction, atr = calc_supertrend(high, low, close, test_p, test_m)
print(f"\nTotal bars: {len(close)}")
print(f"NaN count in ATR: {np.isnan(atr).sum()}")
print(f"Direction changes: {np.sum(np.diff(direction) != 0)}")
print(f"First 20 directions: {direction[:20]}")
print(f"Direction values unique: {np.unique(direction)}")

Result: None

Total bars: 4211
NaN count in ATR: 9
Direction changes: 0
First 20 directions: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
Direction values unique: [1]


In [7]:
def backtest_supertrend_atr(high, low, close,
                             st_period, st_mult,
                             stop_atr_period, stop_mult, target_mult,
                             dd_limit=None):

    st_line, direction, _ = calc_supertrend(high, low, close, st_period, st_mult)

    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i]-low[i], abs(high[i]-close[i-1]), abs(low[i]-close[i-1]))
    risk_atr = pd.Series(tr).rolling(stop_atr_period).mean().values

    trades = []
    pos    = 0
    ep     = 0.0
    stop   = 0.0
    target = 0.0
    cum    = 0.0
    pk     = 0.0
    max_dd = 0.0

    # FIX: warmup must guarantee both ST and risk ATR are valid,
    # and we need i-1 to also be valid so direction comparison works
    warmup = max(st_period, stop_atr_period) + 2

    for i in range(warmup, n):
        if np.isnan(risk_atr[i]) or np.isnan(risk_atr[i-1]):
            continue

        # Check exits first
        if pos == 1:
            st_flip    = (direction[i] == -1)
            hit_target = (high[i] >= target)
            hit_stop   = (low[i] <= stop)

            if st_flip or hit_target or hit_stop:
                if hit_stop:
                    exit_price = stop
                elif hit_target:
                    exit_price = target
                else:
                    exit_price = close[i]

                pnl = (exit_price - ep) * POINT_VALUE
                trades.append(pnl)
                cum += pnl
                pos = 0

        elif pos == -1:
            st_flip    = (direction[i] == 1)
            hit_target = (low[i] <= target)
            hit_stop   = (high[i] >= stop)

            if st_flip or hit_target or hit_stop:
                if hit_stop:
                    exit_price = stop
                elif hit_target:
                    exit_price = target
                else:
                    exit_price = close[i]

                pnl = (ep - exit_price) * POINT_VALUE
                trades.append(pnl)
                cum += pnl
                pos = 0

        # Entry on ST flip when flat
        if pos == 0:
            flipped_bull = (direction[i] == 1 and direction[i-1] == -1)
            flipped_bear = (direction[i] == -1 and direction[i-1] == 1)

            if flipped_bull:
                pos    = 1
                ep     = close[i]
                stop   = ep - stop_mult * risk_atr[i]
                target = ep + target_mult * risk_atr[i]
            elif flipped_bear:
                pos    = -1
                ep     = close[i]
                stop   = ep + stop_mult * risk_atr[i]
                target = ep - target_mult * risk_atr[i]

        if pos == 1:
            opnl = (close[i] - ep) * POINT_VALUE
        elif pos == -1:
            opnl = (ep - close[i]) * POINT_VALUE
        else:
            opnl = 0.0

        run = cum + opnl
        if run > pk:
            pk = run
        dd = run - pk
        if dd < max_dd:
            max_dd = dd
        if dd_limit is not None and max_dd < dd_limit:
            return None

    if len(trades) < 3:
        return None

    pnls = np.array(trades)
    w = pnls[pnls > 0]
    l = pnls[pnls < 0]
    tp = w.sum() if len(w) else 0.0
    tl = abs(l.sum()) if len(l) else 1.0

    return {
        'st_period':   st_period,
        'st_mult':     round(st_mult, 2),
        'stop_atr_p':  stop_atr_period,
        'stop_mult':   round(stop_mult, 2),
        'target_mult': round(target_mult, 2),
        'pf':          round(tp / tl, 4) if tl > 0 else 999.0,
        'total_pnl':   round(float(pnls.sum()), 0),
        'trades':      len(pnls),
        'win_rate':    round(len(w) / len(pnls) * 100, 1),
        'avg_win':     round(float(w.mean()), 0) if len(w) else 0,
        'avg_loss':    round(float(l.mean()), 0) if len(l) else 0,
        'max_dd':      round(float(max_dd), 0),
    }

print("Backtest engine ready (fixed)")

Backtest engine ready (fixed)


In [8]:
t0 = time.time()

# Fixed reasonable stop/target to isolate SuperTrend quality first
FIXED_STOP_ATR_P = 14
FIXED_STOP_MULT  = 1.5
FIXED_TGT_MULT   = 3.0

st_combos = [(p, m) for p in ST_ATR_PERIOD_RANGE for m in ST_MULT_RANGE]
print(f"Stage 1: Testing {len(st_combos)} SuperTrend parameter combinations...")

st_results = []
for i, (p, m) in enumerate(st_combos):
    r = backtest_supertrend_atr(high, low, close, p, m,
                                 FIXED_STOP_ATR_P, FIXED_STOP_MULT, FIXED_TGT_MULT)
    if r:
        st_results.append(r)
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(st_combos)} tested | elapsed {time.time()-t0:.0f}s")

print(f"\nValid results: {len(st_results)} / {len(st_combos)}")

if not st_results:
    print("\n⚠️ STILL NO RESULTS — dataset likely too short or warmup too aggressive")
    print(f"Total bars available: {len(close)}")
    print("Run the diagnostic cell to inspect direction[] and ATR NaN counts")
    best_st_period = None
    best_st_mult   = None
else:
    st_results.sort(key=lambda x: x['pf'], reverse=True)
    print(f"\nTop 10 SuperTrend parameter sets (fixed stop/target):")
    print(f"{'STPeriod':>9} {'STMult':>7} {'PF':>8} {'PnL':>12} {'Trades':>8} {'MaxDD':>12}")
    for r in st_results[:10]:
        print(f"{r['st_period']:>9} {r['st_mult']:>7} {r['pf']:>8.4f} "
              f"${r['total_pnl']:>10,.0f} {r['trades']:>8} ${r['max_dd']:>10,.0f}")

    best_st_period = st_results[0]['st_period']
    best_st_mult   = st_results[0]['st_mult']
    print(f"\nBest SuperTrend: Period={best_st_period}, Mult={best_st_mult}")

print(f"Elapsed: {time.time()-t0:.1f}s")

Stage 1: Testing 390 SuperTrend parameter combinations...
  50/390 tested | elapsed 3s
  100/390 tested | elapsed 5s
  150/390 tested | elapsed 6s
  200/390 tested | elapsed 8s
  250/390 tested | elapsed 12s
  300/390 tested | elapsed 14s
  350/390 tested | elapsed 16s

Valid results: 0 / 390

⚠️ STILL NO RESULTS — dataset likely too short or warmup too aggressive
Total bars available: 4211
Run the diagnostic cell to inspect direction[] and ATR NaN counts
Elapsed: 17.1s


In [9]:
test_p, test_m = 10, 3.0
st_line, direction, atr = calc_supertrend(high, low, close, test_p, test_m)

print(f"Total bars: {len(close)}")
print(f"NaN count in ATR: {np.isnan(atr).sum()}")
print(f"Direction unique values: {np.unique(direction)}")
print(f"Direction changes (flips): {np.sum(np.diff(direction) != 0)}")
print(f"First 15 direction values: {direction[:15]}")
print(f"First 15 ATR values: {atr[:15]}")

Total bars: 4211
NaN count in ATR: 9
Direction unique values: [1]
Direction changes (flips): 0
First 15 direction values: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
First 15 ATR values: [  nan   nan   nan   nan   nan   nan   nan   nan   nan 1.342 1.41  1.21
 1.175 1.07  1.101]


In [10]:
if i == 0 or np.isnan(atr[i]):
    final_upper[i] = raw_upper[i]
    final_lower[i] = raw_lower[i]
    direction[i] = 1

In [11]:
if raw_upper[i] < final_upper[i-1] or close[i-1] > final_upper[i-1]:

SyntaxError: incomplete input (3877401480.py, line 1)